In [1]:
# [Setup]: 

# Ran the following in Anaconda Prompt:
# git clone https://github.com/burke86/deepdisc.git
# cd deepdisc
# conda create -n deepdisc python=3.10 -y
# conda activate deepdisc
# pip install setuptools==67.8.0
# pip install pybind11
# NOTE: scarlet skipped, does not compile on Windows (optional dependency, not needed for detection)

# Ran the following in Anaconda Prompt:
# nvidia-smi

# If Version CUDA is not 12.1, install older versions to be compatible with torch and relevant packages:
# Ran the following in Anaconda Prompt:
# conda activate deepdisc
# pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121

# Ran the following in Anaconda Prompt:
# pip install --no-build-isolation git+https://github.com/facebookresearch/detectron2.git
# mkdir taufit
# echo __version__ = "0.1.0" > taufit\version.py
# echo. > requirements.txt
# pip install --no-build-isolation -e .
# pip install --no-deps --no-build-isolation .
# pip install opencv-python
# pip install scikit-image
# pip install ipympl
# pip install ipywidgets
# pip install astropy photutils matplotlib pandas ipykernel
# python -m ipykernel install --user --name deepdisc --display-name "Python (deepdisc)"

# Now, in VSCodium, click top right for environment -> "Choose another Kernel" -> "deepdisc"

In [2]:
# 02_B_PyTorch_Algorithm.ipynb : Cell 1

%matplotlib widget

import sys
from pathlib import Path
import pickle
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits as astrofits
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm

# Paths
BASE_DIR = Path(r"C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN")

SCRIPTS_DIR = BASE_DIR / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from plate_scan_utils import (
    detect_sources, detect_plate_errors, classify_plate_quality,
    annotate_errors, categorize_plate, fair_plate_features,
)

cutout_dir = BASE_DIR / "data" / "R_CrB" / "cutouts"
manifest_path = cutout_dir.parent / "plate_manifest.csv"

scan_cache_dir = BASE_DIR / "data" / "pytorch_scan"
scan_cache_dir.mkdir(parents=True, exist_ok=True)
PLATE_DB_FILE = scan_cache_dir / "plate_db_pytorch.pkl"
PARADIGM_FILE = scan_cache_dir / "paradigm_marks.pkl"

output_dir = BASE_DIR / "data" / "PyTorch"
img_dir   = output_dir / "images"
mask_dir  = output_dir / "masks"
img_dir.mkdir(parents=True, exist_ok=True)
mask_dir.mkdir(parents=True, exist_ok=True)

cutouts = sorted(list(cutout_dir.glob("*.fits")) + list(cutout_dir.glob("*.fit")))
print(f"Found {len(cutouts)} cutouts to process")
if len(cutouts) == 0:
    raise RuntimeError("No FITS files found in cutout directory")

PRESCAN_WORKERS = 8
QUALITY_VERSION = 1  # bump to force every cached plate to be re-derived

# Plate limiting-magnitude lookup, same source as 02_A
plate_limit_lookup = {}
if manifest_path.exists():
    manifest_df = pd.read_csv(manifest_path)
    if "filename" in manifest_df.columns:
        for _, row in tqdm(manifest_df.iterrows(), total=len(manifest_df), desc="Reading plate manifest", unit="row"):
            fname = row.get("filename")
            if not (isinstance(fname, str) and fname):
                continue
            a, t = row.get("lim_mag_apass"), row.get("lim_mag_atlas")
            plate_limit_lookup[Path(fname).name] = {
                "lim_mag_apass": float(a) if pd.notna(a) else None,
                "lim_mag_atlas": float(t) if pd.notna(t) else None,
            }
print(f"Loaded plate limits for {len(plate_limit_lookup)} manifest rows.")

def get_plate_limits(fits_path):
    entry = plate_limit_lookup.get(fits_path.name)
    return (entry["lim_mag_apass"], entry["lim_mag_atlas"]) if entry else (None, None)

_apass_vals = [v["lim_mag_apass"] for v in plate_limit_lookup.values() if v.get("lim_mag_apass") is not None]
_atlas_vals = [v["lim_mag_atlas"] for v in plate_limit_lookup.values() if v.get("lim_mag_atlas") is not None]
LIM_MAG_MEDIAN_APASS = float(np.median(_apass_vals)) if _apass_vals else None
LIM_MAG_MEDIAN_ATLAS = float(np.median(_atlas_vals)) if _atlas_vals else None

# Plate database (source detections + quality only -- no Gaia/target info)
if PLATE_DB_FILE.exists():
    with open(PLATE_DB_FILE, "rb") as fp:
        plate_db = pickle.load(fp)
else:
    plate_db = {}

def _scan_one_plate(f):
    cached = plate_db.get(str(f))
    if cached is not None and cached.get("quality_version") == QUALITY_VERSION:
        return (f, "skip", None)
    try:
        sources, algorithm, data, data_sub, std, x_col, y_col = detect_sources(f)
        n = 0 if sources is None else len(sources)
        errors = detect_plate_errors(data if sources is not None else astrofits.getdata(f).astype(float))
        lim_apass, lim_atlas = get_plate_limits(f)
        quality = classify_plate_quality(
            errors, n, None, lim_apass, lim_atlas, LIM_MAG_MEDIAN_APASS, LIM_MAG_MEDIAN_ATLAS
        )
        entry = {
            "n": n, "quality": quality, "quality_version": QUALITY_VERSION,
            "errors": errors, "sources": sources, "x_col": x_col, "y_col": y_col,
            "human_verdict": cached.get("human_verdict") if cached else None,
        }
        return (f, "new", entry)
    except Exception as e:
        entry = {
            "n": 0, "quality": "defective", "quality_version": QUALITY_VERSION,
            "errors": {}, "sources": None, "x_col": None, "y_col": None,
            "human_verdict": None, "error": str(e),
        }
        return (f, "new", entry)

def prescan(plates=None, max_workers=PRESCAN_WORKERS):
    target_plates = plates if plates is not None else cutouts
    total = max(len(target_plates), 1)
    errors_seen = []

    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {ex.submit(_scan_one_plate, f): f for f in target_plates}
        pbar = tqdm(as_completed(futures), total=total, desc="Prescanning plates", unit="plate")
        for fut in pbar:
            f = futures[fut]
            try:
                f_ret, action, payload = fut.result()
            except Exception as e:
                errors_seen.append(f"{f.name}: {e}")
                continue
            if action == "new":
                plate_db[str(f)] = payload
                if "error" in payload:
                    errors_seen.append(f"{f.name}: {payload['error']}")
            pbar.set_postfix(cached=len(plate_db))

    with open(PLATE_DB_FILE, "wb") as fp:
        pickle.dump(plate_db, fp)

    print(f"Prescan complete: {len(plate_db)} plates cached.")
    if errors_seen:
        print(f"{len(errors_seen)} plate(s) had errors during scanning:")
        for msg in errors_seen[:20]:
            print(f"  [SCAN ERROR] {msg}")
        if len(errors_seen) > 20:
            print(f"  ...and {len(errors_seen) - 20} more.")

prescan()

Found 10803 cutouts to process


Reading plate manifest:   0%|          | 0/15217 [00:00<?, ?row/s]

Loaded plate limits for 10662 manifest rows.


Prescanning plates:   0%|          | 0/10803 [00:00<?, ?plate/s]

Prescan complete: 10803 plates cached.


In [3]:
# 02_B_PyTorch_Algorithm.ipynb : Cell 2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import cv2
from tqdm.notebook import tqdm
import random

BOX_HALF = 8
VAL_SPLIT = 0.15
EPOCHS = 10

def crop_to_match(x, ref):
    _, _, h, w = ref.shape
    return x[:, :, :h, :w]

def pad_to_multiple(img, multiple=16):
    _, h, w = img.shape
    pad_h = (multiple - h % multiple) % multiple
    pad_w = (multiple - w % multiple) % multiple
    return F.pad(img, (0, pad_w, 0, pad_h))

class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        def block(in_c, out_c):
            return nn.Sequential(nn.Conv2d(in_c, out_c, 3, padding=1), nn.ReLU(),
                                  nn.Conv2d(out_c, out_c, 3, padding=1), nn.ReLU())
        self.enc1, self.enc2, self.enc3 = block(1, 32), block(32, 64), block(64, 128)
        self.pool = nn.MaxPool2d(2)
        self.mid = block(128, 256)
        self.up3, self.dec3 = nn.ConvTranspose2d(256, 128, 2, stride=2), block(256, 128)
        self.up2, self.dec2 = nn.ConvTranspose2d(128, 64, 2, stride=2), block(128, 64)
        self.up1, self.dec1 = nn.ConvTranspose2d(64, 32, 2, stride=2), block(64, 32)
        self.out = nn.Conv2d(32, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        m = self.mid(self.pool(e3))
        d3 = self.dec3(torch.cat([self.up3(m), crop_to_match(e3, self.up3(m))], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), crop_to_match(e2, self.up2(d3))], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), crop_to_match(e1, self.up1(d2))], dim=1))
        return torch.sigmoid(self.out(d1))

class StarSegDataset(Dataset):
    def __init__(self, image_paths, mask_paths):
        self.image_paths, self.mask_paths = image_paths, mask_paths

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = cv2.imread(str(self.image_paths[idx]), cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255.0
        mask = cv2.imread(str(self.mask_paths[idx]), cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255.0
        img = pad_to_multiple(torch.tensor(img).unsqueeze(0))
        mask = pad_to_multiple(torch.tensor(mask).unsqueeze(0))
        return img, mask

def _points_to_mask(shape, xs, ys, box_half=BOX_HALF):
    h, w = shape
    mask = np.zeros((h, w), dtype=np.uint8)
    for x, y in zip(xs, ys):
        x0, x1 = max(0, int(x - box_half)), min(w, int(x + box_half))
        y0, y1 = max(0, int(y - box_half)), min(h, int(y + box_half))
        mask[y0:y1, x0:x1] = 255
    return mask

def _save_image_mask_pair(fits_path, xs, ys, out_img_dir, out_mask_dir):
    data = astrofits.getdata(fits_path).astype(float)
    data = np.nan_to_num(data, nan=np.nanmedian(data))
    vmin, vmax = np.percentile(data, [1, 99])
    img_norm = np.clip((data - vmin) / (vmax - vmin + 1e-8), 0, 1)
    img_8bit = (img_norm * 255).astype(np.uint8)
    mask = _points_to_mask(img_8bit.shape, xs, ys)

    img_path = out_img_dir / (fits_path.stem + ".png")
    mask_path = out_mask_dir / (fits_path.stem + "_mask.png")
    cv2.imwrite(str(img_path), img_8bit)
    cv2.imwrite(str(mask_path), mask)
    return img_path, mask_path

def build_dataset_and_train():
    image_paths, mask_paths = [], []

    # 1. Ground truth: your human-corrected paradigm plates
    paradigm_items = [(f_str, marks) for f_str, marks in paradigm_marks.items() if marks]
    for f_str, marks in tqdm(paradigm_items, desc="Building paradigm (ground-truth) plates", unit="plate"):
        fp = Path(f_str)
        xs, ys = [m[0] for m in marks], [m[1] for m in marks]
        ip, mp = _save_image_mask_pair(fp, xs, ys, img_dir, mask_dir)
        image_paths.append(ip); mask_paths.append(mp)
    print(f"{len(image_paths)} paradigm (ground-truth) plate(s) added.")

    # 2. Pseudo-labels: every other cached plate that passed the quality
    #    bar (algorithm-detected sources used as weak labels), skipping
    #    anything defective/too_many_errors or explicitly rejected, and
    #    skipping plates already added above.
    pseudo_items = [
        (f_str, meta) for f_str, meta in plate_db.items()
        if not (f_str in paradigm_marks and paradigm_marks[f_str])
        and meta.get("human_verdict") != "rejected"
        and meta.get("quality") in ("good_match", "fair")
        and meta.get("sources") is not None
    ]
    n_pseudo = 0
    for f_str, meta in tqdm(pseudo_items, desc="Building pseudo-labeled plates", unit="plate"):
        fp = Path(f_str)
        xs = list(meta["sources"][meta["x_col"]])
        ys = list(meta["sources"][meta["y_col"]])
        ip, mp = _save_image_mask_pair(fp, xs, ys, img_dir, mask_dir)
        image_paths.append(ip); mask_paths.append(mp)
        n_pseudo += 1
    print(f"{n_pseudo} pseudo-labeled plate(s) added (quality-filtered algorithm detections).")

    if len(image_paths) < 4:
        print("Too few plates to train on -- label more stars on your paradigm plate(s) or approve more 'fair' plates.")
        return

    combined = list(zip(image_paths, mask_paths))
    random.shuffle(combined)
    n_val = max(1, int(len(combined) * VAL_SPLIT))
    val_pairs, train_pairs = combined[:n_val], combined[n_val:]

    train_ds = StarSegDataset([p[0] for p in train_pairs], [p[1] for p in train_pairs])
    val_ds = StarSegDataset([p[0] for p in val_pairs], [p[1] for p in val_pairs])
    train_loader = DataLoader(train_ds, batch_size=2, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=2)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = UNet().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    loss_fn = nn.BCELoss()

    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        for imgs, masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]"):
            imgs, masks = imgs.to(device), masks.to(device)
            preds = model(imgs)
            loss = loss_fn(preds, masks)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            train_loss += loss.item()

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for imgs, masks in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]"):
                imgs, masks = imgs.to(device), masks.to(device)
                val_loss += loss_fn(model(imgs), masks).item()

        print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f} | Val Loss={val_loss:.4f}")

    model_path = BASE_DIR / "data" / "PyTorch" / "pytorch_trial_1.pth"
    torch.save(model.state_dict(), model_path)
    print(f"Training complete. Model saved to {model_path}")

In [4]:
# 02_B_PyTorch_Algorithm.ipynb : Cell 3

import ipywidgets as widgets
from IPython.display import display, clear_output
from io import BytesIO

REVIEW_CLUSTERS = 20  # hard cap on representative plates shown for review

if PARADIGM_FILE.exists():
    with open(PARADIGM_FILE, "rb") as f:
        paradigm_marks = pickle.load(f)  # { plate_path_str: [ [x,y], ... ] }
else:
    paradigm_marks = {}

def save_paradigm_marks():
    with open(PARADIGM_FILE, "wb") as f:
        pickle.dump(paradigm_marks, f)

# ---------- Review queue (clustered "fair" plates, capped at 20) ----------

_fair_cluster_of = {}
_fair_cluster_members = {}

def _cluster_fair_plates(n_clusters=REVIEW_CLUSTERS):
    global _fair_cluster_of, _fair_cluster_members
    _fair_cluster_of, _fair_cluster_members = {}, {}

    fair_items = [
        (f_str, meta) for f_str, meta in plate_db.items()
        if meta.get("quality") == "fair" and meta.get("human_verdict") is None
        and meta.get("sources") is not None
    ]
    if not fair_items:
        return []
    if len(fair_items) <= 1:
        f_str = fair_items[0][0]
        _fair_cluster_of[f_str] = 0
        _fair_cluster_members[0] = [f_str]
        return [f_str]

    feats = np.array([
        fair_plate_features(
            meta["errors"], meta["n"], *get_plate_limits(Path(f_str)),
            LIM_MAG_MEDIAN_APASS, LIM_MAG_MEDIAN_ATLAS
        )
        for f_str, meta in fair_items
    ])
    means, stds = feats.mean(axis=0), feats.std(axis=0)
    stds[stds < 1e-9] = 1.0
    feats_z = np.nan_to_num((feats - means) / stds)

    k = max(1, min(n_clusters, len(fair_items)))
    try:
        from scipy.cluster.vq import kmeans2
        _, labels = kmeans2(feats_z, k, minit="++", seed=0)
    except Exception as e:
        print(f"[CLUSTERING] {e} -- falling back to round-robin")
        labels = np.arange(len(fair_items)) % k

    reps = []
    for cid in range(k):
        idx = np.where(labels == cid)[0]
        if len(idx) == 0:
            continue
        members = [fair_items[i][0] for i in idx]
        for m in members:
            _fair_cluster_of[m] = cid
        _fair_cluster_members[cid] = members
        centroid = feats_z[idx].mean(axis=0)
        rep_idx = idx[int(np.argmin(np.linalg.norm(feats_z[idx] - centroid, axis=1)))]
        reps.append(fair_items[rep_idx][0])
    return reps

review_dropdown = widgets.Dropdown(description="Needs review:")
review_status = widgets.Label(value="")
review_output = widgets.Output()
approve_btn = widgets.Button(description="Approve", button_style="success")
reject_btn = widgets.Button(description="Reject", button_style="danger")

def _refresh_review_queue():
    reps = _cluster_fair_plates()
    options = []
    for f_str in reps:
        meta = plate_db[f_str]
        cid = _fair_cluster_of.get(f_str)
        n_members = len(_fair_cluster_members.get(cid, [f_str]))
        label = f"{Path(f_str).name} | represents {n_members} plate(s) | {meta['n']} sources"
        options.append((label, Path(f_str)))
    review_dropdown.options = options
    review_status.value = f"{len(options)} representative plate(s) awaiting review (capped at {REVIEW_CLUSTERS})."

def _open_review_plate(change):
    with review_output:
        clear_output(wait=True)
        fp = review_dropdown.value
        if fp is None:
            print("Nothing to review.")
            return
        meta = plate_db[str(fp)]
        data = astrofits.getdata(fp)
        fig, ax = plt.subplots(figsize=(6, 6))
        ax.imshow(data, origin="lower", cmap="gray")
        if meta.get("sources") is not None:
            ax.scatter(meta["sources"][meta["x_col"]], meta["sources"][meta["y_col"]], s=30, facecolors="none", edgecolors="red")
        annotate_errors(ax, meta["errors"])
        ax.set_title(fp.name)
        buf = BytesIO(); fig.savefig(buf, format="png", bbox_inches="tight"); plt.close(fig); buf.seek(0)
        display(widgets.Image(value=buf.read(), format="png"))

def _approve(_):
    fp = review_dropdown.value
    if fp is None: return
    for m in _fair_cluster_members.get(_fair_cluster_of.get(str(fp)), [str(fp)]):
        plate_db[m]["human_verdict"] = "approved"
    with open(PLATE_DB_FILE, "wb") as f: pickle.dump(plate_db, f)
    _refresh_review_queue()
    rebuild_paradigm_dropdown()

def _reject(_):
    fp = review_dropdown.value
    if fp is None: return
    for m in _fair_cluster_members.get(_fair_cluster_of.get(str(fp)), [str(fp)]):
        plate_db[m]["human_verdict"] = "rejected"
    with open(PLATE_DB_FILE, "wb") as f: pickle.dump(plate_db, f)
    _refresh_review_queue()
    rebuild_paradigm_dropdown()

review_dropdown.observe(_open_review_plate, names="value")
approve_btn.on_click(_approve)
reject_btn.on_click(_reject)

display(widgets.VBox([
    widgets.HTML("<b>Review uncertain plates</b> (approve = treat as good, reject = exclude from paradigm picker)"),
    review_dropdown, review_status, widgets.HBox([approve_btn, reject_btn]), review_output,
]))
_refresh_review_queue()

# ---------- Paradigm plate filter + selection ----------

category_filter = widgets.Dropdown(options=["all", "ideal", "good_no_target", "defective_no_target"], value="all", description="Filter:")
search_box = widgets.Text(description="Search:", placeholder="filter by filename")
paradigm_dropdown = widgets.Dropdown(description="Paradigm plate:")
paradigm_dropdown_status = widgets.Label(value="")

def filter_plates(cat_value, query):
    query = query.strip().lower()
    matches = []
    for f_str, meta in plate_db.items():
        if meta.get("human_verdict") == "rejected":
            continue
        cat = categorize_plate(meta)
        if cat_value != "all" and cat != cat_value:
            continue
        f = Path(f_str)
        if query and query not in f.name.lower():
            continue
        label = f"{f.name} | {cat} | {meta['n']} sources"
        matches.append((label, f))
    return matches

def rebuild_paradigm_dropdown(*args):
    matches = filter_plates(category_filter.value, search_box.value)
    paradigm_dropdown.options = matches
    paradigm_dropdown_status.value = f"{len(matches)} matches"

category_filter.observe(rebuild_paradigm_dropdown, names="value")
search_box.observe(rebuild_paradigm_dropdown, names="value")
rebuild_paradigm_dropdown()

# ---------- Paradigm marking (click to add/remove source markers) ----------

PARADIGM_CLICK_PX = 15
_pstate = {"fig": None, "ax": None, "scatter": None, "fits_path": None, "xs": [], "ys": []}

paradigm_plot_output = widgets.Output()
paradigm_msg = widgets.Output()
auto_load_btn = widgets.Button(description="Auto-Load Stars", button_style="info")
clear_btn = widgets.Button(description="Clear Plate", button_style="danger")
train_btn = widgets.Button(description="Build Training Set & Train Model", button_style="success")
train_status = widgets.Label(value="")

def _nearest(x, y, max_px=PARADIGM_CLICK_PX):
    if not _pstate["xs"]:
        return None
    d = np.hypot(np.array(_pstate["xs"]) - x, np.array(_pstate["ys"]) - y)
    j = int(np.argmin(d))
    return j if d[j] <= max_px else None

def _redraw():
    ax = _pstate["ax"]
    if _pstate["scatter"] is not None:
        _pstate["scatter"].remove()
    _pstate["scatter"] = ax.scatter(_pstate["xs"], _pstate["ys"], s=50, facecolors="none", edgecolors="orange", linewidths=1.5)
    ax.figure.canvas.draw_idle()

def _on_click(event):
    if event.inaxes != _pstate["ax"] or event.xdata is None:
        return
    if event.button == 3:  # right-click = remove
        j = _nearest(event.xdata, event.ydata)
        if j is not None:
            _pstate["xs"].pop(j); _pstate["ys"].pop(j)
            _redraw()
    else:  # left-click = add
        _pstate["xs"].append(float(event.xdata))
        _pstate["ys"].append(float(event.ydata))
        _redraw()
    paradigm_marks[str(_pstate["fits_path"])] = list(zip(_pstate["xs"], _pstate["ys"]))
    save_paradigm_marks()

def _open_paradigm(change):
    fp = paradigm_dropdown.value
    if fp is None:
        return
    data = astrofits.getdata(fp)
    marks = paradigm_marks.get(str(fp), [])
    _pstate.update({"fits_path": fp, "xs": [m[0] for m in marks], "ys": [m[1] for m in marks], "scatter": None})
    with paradigm_plot_output:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(7, 7))
        ax.imshow(data, origin="lower", cmap="gray")
        ax.set_title(f"{fp.name}  (left-click=add star, right-click=remove)")
        fig.canvas.mpl_connect("button_press_event", _on_click)
        _pstate["fig"], _pstate["ax"] = fig, ax
        _redraw()
        plt.show()

def _auto_load(_):
    fp = _pstate["fits_path"]
    if fp is None:
        with paradigm_msg:
            clear_output(wait=True); print("Open a paradigm plate first.")
        return
    meta = plate_db.get(str(fp))
    if meta is None or meta.get("sources") is None:
        with paradigm_msg:
            clear_output(wait=True); print("No cached detections for this plate.")
        return
    new_xs = np.array(meta["sources"][meta["x_col"]], dtype=float)
    new_ys = np.array(meta["sources"][meta["y_col"]], dtype=float)
    added = 0
    for nx, ny in zip(new_xs, new_ys):
        if _pstate["xs"]:
            d = np.hypot(np.array(_pstate["xs"]) - nx, np.array(_pstate["ys"]) - ny)
            if d.min() <= PARADIGM_CLICK_PX:
                continue
        _pstate["xs"].append(float(nx)); _pstate["ys"].append(float(ny))
        added += 1
    _redraw()
    paradigm_marks[str(fp)] = list(zip(_pstate["xs"], _pstate["ys"]))
    save_paradigm_marks()
    with paradigm_msg:
        clear_output(wait=True); print(f"Auto-loaded {added} detection(s). Remove false positives with right-click.")

def _clear_plate(_):
    fp = _pstate["fits_path"]
    if fp is None:
        return
    _pstate["xs"], _pstate["ys"] = [], []
    _redraw()
    paradigm_marks.pop(str(fp), None)
    save_paradigm_marks()
    with paradigm_msg:
        clear_output(wait=True); print("Cleared all markers for this plate.")

def _on_train_click(_):
    n_marked = sum(1 for v in paradigm_marks.values() if len(v) > 0)
    if n_marked == 0:
        train_status.value = "Mark at least one star on a paradigm plate first."
        return
    train_status.value = "Building training set and training model... (see output below)"
    build_dataset_and_train()  # defined in Cell 2 -- run Cell 2 before clicking this
    train_status.value = "Done -- see training log above."

auto_load_btn.on_click(_auto_load)
clear_btn.on_click(_clear_plate)
train_btn.on_click(_on_train_click)
paradigm_dropdown.observe(_open_paradigm, names="value")

display(widgets.VBox([
    widgets.HTML("<b>Paradigm plate labeling</b> -- pick a plate, mark true star positions, then train."),
    category_filter, search_box, paradigm_dropdown, paradigm_dropdown_status,
    widgets.HBox([auto_load_btn, clear_btn]),
    paradigm_plot_output, paradigm_msg,
    train_btn, train_status,
]))